# CIFAR-100 Full Training Pipeline

This notebook mirrors the CIFAR-10 workflow in `full_training_pipeline.ipynb` and applies it to CIFAR-100 by reusing the same configs with overrides:

- `data.name=cifar100`
- `model.num_classes=100`
- CIFAR-100-specific `experiment.name` values and output directories

The training split is `45k train / 5k val` from the original CIFAR-100 training split. The official 10k test split is evaluated once on `best.pth` and reported as `test_acc1` and `test_acc5`.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "tools" / "train.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "tools" / "train.py").exists(), f"Run this notebook from the repo root or notebooks dir. Got {ROOT}"

COC_PYTHON = Path(r"D:\Anaconda\envs\CoC\python.exe")
PYTHON = COC_PYTHON if COC_PYTHON.exists() else Path(sys.executable)

print("Repo:", ROOT)
print("Python:", PYTHON)

## Phase Switches

Use the same execution style as the CIFAR-10 notebook: start with checks and smoke, then enable one research phase at a time.

In [ ]:
RUN_ENV_CHECKS = True
RUN_SMOKE = True

RUN_TEACHER = False
RUN_LIGHTWEIGHT_BASELINES = False
RUN_COC_BASELINE = False
RUN_STUDENT_CE = False
RUN_PROXY_SEARCH = False
RUN_ABLATIONS = False
RUN_KD = False
RUN_PRUNING = False
RUN_BENCHMARKS = False
RUN_SUMMARY_REPORT = False
RUN_PARETO_REPORT = False

FULL_EPOCHS = 200
ACCURACY_EPOCHS = 300
PROXY_EPOCHS = 30
PRUNING_EPOCHS = 80
PRUNED_FINETUNE_EPOCHS = 80

CIFAR100_OVERRIDES = [
    "data.name=cifar100",
    "model.num_classes=100",
]

COMMON_TRAIN_OVERRIDES = [
    # Reduce these if you hit OOM.
    # "data.batch_size=64",
    # "data.val_batch_size=128",
    # "data.test_batch_size=128",
]

BENCHMARK_OUTPUT = "results/benchmark_cifar100"
BENCHMARK_BATCHES = [1, 16, 64, 128]
BENCHMARK_WARMUP = 30
BENCHMARK_RUNS = 100

DEBUG_BENCHMARK_BATCHES = [1, 16]
DEBUG_BENCHMARK_WARMUP = 2
DEBUG_BENCHMARK_RUNS = 3

In [ ]:
def _looks_like_tqdm(line: str) -> bool:
    text = line.strip()
    return "%|" in text or text.startswith("eval:") or text.startswith("train ")


def run_py(args: list[str], check: bool = True) -> int:
    cmd = [str(PYTHON), *args]
    print("\n$", " ".join(cmd))
    start = time.perf_counter()
    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    assert process.stdout is not None
    progress_handle = display(Markdown(""), display_id=True)
    last_progress = ""
    for line in process.stdout:
        clean = line.rstrip("\r\n")
        if _looks_like_tqdm(clean):
            last_progress = clean
            progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
        else:
            print(line, end="")
    code = process.wait()
    elapsed = time.perf_counter() - start
    if last_progress:
        progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
    print(f"\n[exit={code}] elapsed={elapsed:.1f}s")
    if check and code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {' '.join(cmd)}")
    return code


def override_args(overrides: list[str] | None) -> list[str]:
    args: list[str] = []
    for item in overrides or []:
        args.extend(["--override", item])
    return args


def cifar100_overrides(experiment_name: str | None = None, overrides: list[str] | None = None) -> list[str]:
    items = [*CIFAR100_OVERRIDES]
    if experiment_name is not None:
        items.append(f"experiment.name={experiment_name}")
    items.extend(overrides or [])
    return items


def train(
    config: str,
    output: str,
    experiment_name: str,
    overrides: list[str] | None = None,
    extra: list[str] | None = None,
) -> None:
    args = ["tools/train.py", "--config", config, "--output", output]
    args += override_args(cifar100_overrides(experiment_name, [*(overrides or []), *COMMON_TRAIN_OVERRIDES]))
    args += extra or []
    run_py(args)


def benchmark(
    config: str,
    experiment_name: str,
    checkpoint: str | None = None,
    debug: bool = False,
    profile: bool = True,
) -> None:
    batches = DEBUG_BENCHMARK_BATCHES if debug else BENCHMARK_BATCHES
    warmup = DEBUG_BENCHMARK_WARMUP if debug else BENCHMARK_WARMUP
    runs = DEBUG_BENCHMARK_RUNS if debug else BENCHMARK_RUNS
    args = [
        "tools/benchmark.py",
        "--config",
        config,
        "--output",
        BENCHMARK_OUTPUT,
        "--batch-sizes",
        *[str(v) for v in batches],
        "--warmup",
        str(warmup),
        "--runs",
        str(runs),
    ]
    args += override_args(cifar100_overrides(experiment_name))
    if checkpoint and Path(ROOT / checkpoint).exists():
        args += ["--checkpoint", checkpoint]
    elif checkpoint:
        print(f"Skip checkpoint for {config}; not found: {checkpoint}")
    if profile:
        args.append("--profile")
    run_py(args)

## 0. Environment And Smoke Checks

In [ ]:
if RUN_ENV_CHECKS:
    run_py(["-m", "pytest"])
    run_py(["tools/shape_trace.py", "--config", "configs/hbcc_latency_tiny.yaml", "--batch-size", "1", *override_args(CIFAR100_OVERRIDES)])
    run_py(["tools/shape_trace.py", "--config", "configs/hbcc_current_reference.yaml", "--batch-size", "1", *override_args(CIFAR100_OVERRIDES)])

if RUN_SMOKE:
    run_py([
        "tools/train.py",
        "--config",
        "configs/smoke.yaml",
        "--output",
        "runs_smoke_cifar100",
        "--override",
        "experiment.name=smoke_hbcc_latency_tiny_fake_cifar100",
        "--override",
        "model.num_classes=100",
        "--limit-train-batches",
        "1",
        "--limit-val-batches",
        "1",
        "--limit-test-batches",
        "1",
    ])
    benchmark("configs/smoke.yaml", "smoke_hbcc_latency_tiny_fake_cifar100", debug=True, profile=False)

## 1. Teacher And Reference Baselines

In [ ]:
if RUN_TEACHER:
    train(
        "configs/baselines/resnet18_cifar.yaml",
        "runs_teacher_cifar100",
        "resnet18_cifar100",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )

if RUN_LIGHTWEIGHT_BASELINES:
    train(
        "configs/baselines/mobilenet_v2_cifar.yaml",
        "runs_baselines_cifar100",
        "mobilenet_v2_cifar100",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )
    train(
        "configs/baselines/shufflenet_v2_x1_0_cifar.yaml",
        "runs_baselines_cifar100",
        "shufflenet_v2_x1_0_cifar100",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )

## 2. CoC Baseline

In [ ]:
COC_BASELINE_RUN = "runs_coc_baseline_cifar100/coc_cifar_baseline_cifar100"
COC_BASELINE_CHECKPOINT = f"{COC_BASELINE_RUN}/best.pth"

if RUN_COC_BASELINE:
    train(
        "configs/coc_cifar_baseline.yaml",
        "runs_coc_baseline_cifar100",
        "coc_cifar_baseline_cifar100",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )
    run_py(["tools/summarize_training.py", COC_BASELINE_RUN])

## 3. Student CE-Only Training

In [ ]:
if RUN_STUDENT_CE:
    train(
        "configs/hbcc_latency_tiny.yaml",
        "runs_students_cifar100",
        "hbcc_latency_tiny_cifar100",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )
    train(
        "configs/hbcc_latency_small.yaml",
        "runs_students_cifar100",
        "hbcc_latency_small_cifar100",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )
    train(
        "configs/hbcc_accuracy_small.yaml",
        "runs_accuracy_cifar100",
        "hbcc_accuracy_small_cifar100",
        overrides=[f"train.epochs={ACCURACY_EPOCHS}"],
    )
    train(
        "configs/hbcc_accuracy_medium.yaml",
        "runs_accuracy_cifar100",
        "hbcc_accuracy_medium_cifar100",
        overrides=[f"train.epochs={ACCURACY_EPOCHS}"],
    )

## 4. Short Proxy Architecture Search

In [ ]:
PROXY_JOBS = [
    {
        "name": "proxy_dims24_depth1111_mlp2_cifar100",
        "overrides": [
            "model.embed_dims=[24,48,96,160]",
            "model.depths=[1,1,1,1]",
            "model.mlp_ratios=2.0",
        ],
    },
    {
        "name": "proxy_dims32_depth1121_mlp2_cifar100",
        "overrides": [
            "model.embed_dims=[32,48,96,160]",
            "model.depths=[1,1,2,1]",
            "model.mlp_ratios=2.0",
        ],
    },
    {
        "name": "proxy_dims32_64_depth1221_mlp3_cifar100",
        "overrides": [
            "model.embed_dims=[32,64,128,192]",
            "model.depths=[1,2,2,1]",
            "model.mlp_ratios=3.0",
            "model.heads=[1,2,4,4]",
        ],
    },
]

if RUN_PROXY_SEARCH:
    for job in PROXY_JOBS:
        train(
            "configs/hbcc_latency_tiny.yaml",
            "runs_proxy_cifar100",
            job["name"],
            overrides=[
                f"train.epochs={PROXY_EPOCHS}",
                *job["overrides"],
            ],
        )

## 5. Independent Ablations

In [ ]:
ABLATION_CONFIGS = [
    ("configs/ablations/hbcc_tiny_hamming_late.yaml", "hbcc_tiny_hamming_late_cifar100"),
    ("configs/ablations/hbcc_tiny_lbpconv.yaml", "hbcc_tiny_lbpconv_cifar100"),
]

if RUN_ABLATIONS:
    for cfg, experiment_name in ABLATION_CONFIGS:
        train(cfg, "runs_ablations_cifar100", experiment_name, overrides=[f"train.epochs={FULL_EPOCHS}"])

## 6. Knowledge Distillation

In [ ]:
TEACHER_CHECKPOINT = "runs_teacher_cifar100/resnet18_cifar100/best.pth"

if RUN_KD:
    if not (ROOT / TEACHER_CHECKPOINT).exists():
        raise FileNotFoundError(f"Train the teacher first: {TEACHER_CHECKPOINT}")
    train(
        "configs/hbcc_latency_tiny.yaml",
        "runs_kd_cifar100",
        "hbcc_latency_tiny_kd_cifar100",
        overrides=[
            f"train.epochs={FULL_EPOCHS}",
            "train.kd_alpha=0.5",
            "train.kd_temperature=4.0",
        ],
        extra=[
            "--teacher-config",
            "configs/baselines/resnet18_cifar.yaml",
            "--teacher-checkpoint",
            TEACHER_CHECKPOINT,
        ],
    )

## 7. Structured Pruning Export

In [ ]:
PRUNING_BASE_CHECKPOINT = "runs_accuracy_cifar100/hbcc_accuracy_small_cifar100/best.pth"
PRUNING_CONFIG = "configs/ablations/hbcc_accuracy_small_pruning_mask.yaml"
PRUNING_RUN_DIR = "runs_pruning_accuracy_cifar100"
PRUNING_EXPERIMENT = "hbcc_accuracy_small_pruning_mask_cifar100"
PRUNING_CHECKPOINT = f"{PRUNING_RUN_DIR}/{PRUNING_EXPERIMENT}/best.pth"
PRUNING_THRESHOLD = "0.90"
PRUNED_CONFIG = "configs/generated_ablations/hbcc_accuracy_small_cifar100_pruned_export.yaml"
PRUNED_FINETUNE_RUN_DIR = "runs_pruned_accuracy_cifar100"
PRUNED_FINETUNE_EXPERIMENT = "hbcc_accuracy_small_cifar100_pruned_export"
PRUNED_FINETUNE_CHECKPOINT = f"{PRUNED_FINETUNE_RUN_DIR}/{PRUNED_FINETUNE_EXPERIMENT}/best.pth"

if RUN_PRUNING:
    if not (ROOT / PRUNING_BASE_CHECKPOINT).exists():
        raise FileNotFoundError(f"Train hbcc_accuracy_small first: {PRUNING_BASE_CHECKPOINT}")
    train(
        PRUNING_CONFIG,
        PRUNING_RUN_DIR,
        PRUNING_EXPERIMENT,
        overrides=[f"train.epochs={PRUNING_EPOCHS}"],
        extra=["--resume", PRUNING_BASE_CHECKPOINT],
    )
    run_py([
        "tools/inspect_pruning_masks.py",
        "--config",
        PRUNING_CONFIG,
        "--checkpoint",
        PRUNING_CHECKPOINT,
        "--thresholds",
        "0.5",
        "0.7",
        "0.8",
        "0.9",
        "0.95",
        *override_args(cifar100_overrides(PRUNING_EXPERIMENT)),
    ])
    run_py([
        "tools/export_pruned.py",
        "--config",
        PRUNING_CONFIG,
        "--checkpoint",
        PRUNING_CHECKPOINT,
        "--output",
        PRUNED_CONFIG,
        "--threshold",
        PRUNING_THRESHOLD,
        "--min-channels",
        "16",
        "--round-to",
        "8",
        *override_args(cifar100_overrides(PRUNED_FINETUNE_EXPERIMENT)),
    ])
    train(
        PRUNED_CONFIG,
        PRUNED_FINETUNE_RUN_DIR,
        PRUNED_FINETUNE_EXPERIMENT,
        overrides=[f"train.epochs={PRUNED_FINETUNE_EPOCHS}"],
    )

## 8. Benchmark Matrix

In [ ]:
BENCHMARK_JOBS = [
    ("configs/baselines/resnet18_cifar.yaml", "resnet18_cifar100", "runs_teacher_cifar100/resnet18_cifar100/best.pth"),
    ("configs/baselines/mobilenet_v2_cifar.yaml", "mobilenet_v2_cifar100", "runs_baselines_cifar100/mobilenet_v2_cifar100/best.pth"),
    ("configs/baselines/shufflenet_v2_x1_0_cifar.yaml", "shufflenet_v2_x1_0_cifar100", "runs_baselines_cifar100/shufflenet_v2_x1_0_cifar100/best.pth"),
    ("configs/coc_cifar_baseline.yaml", "coc_cifar_baseline_cifar100", COC_BASELINE_CHECKPOINT),
    ("configs/hbcc_current_reference.yaml", "hbcc_current_reference_cifar100", None),
    ("configs/hbcc_latency_tiny.yaml", "hbcc_latency_tiny_cifar100", "runs_students_cifar100/hbcc_latency_tiny_cifar100/best.pth"),
    ("configs/hbcc_latency_small.yaml", "hbcc_latency_small_cifar100", "runs_students_cifar100/hbcc_latency_small_cifar100/best.pth"),
    ("configs/hbcc_accuracy_small.yaml", "hbcc_accuracy_small_cifar100", "runs_accuracy_cifar100/hbcc_accuracy_small_cifar100/best.pth"),
    ("configs/hbcc_accuracy_medium.yaml", "hbcc_accuracy_medium_cifar100", "runs_accuracy_cifar100/hbcc_accuracy_medium_cifar100/best.pth"),
    ("configs/hbcc_latency_tiny.yaml", "hbcc_latency_tiny_kd_cifar100", "runs_kd_cifar100/hbcc_latency_tiny_kd_cifar100/best.pth"),
    ("configs/ablations/hbcc_tiny_hamming_late.yaml", "hbcc_tiny_hamming_late_cifar100", "runs_ablations_cifar100/hbcc_tiny_hamming_late_cifar100/best.pth"),
    ("configs/ablations/hbcc_tiny_lbpconv.yaml", "hbcc_tiny_lbpconv_cifar100", "runs_ablations_cifar100/hbcc_tiny_lbpconv_cifar100/best.pth"),
    (PRUNED_CONFIG, PRUNED_FINETUNE_EXPERIMENT, PRUNED_FINETUNE_CHECKPOINT),
]

if RUN_BENCHMARKS:
    for cfg, experiment_name, ckpt in BENCHMARK_JOBS:
        if not (ROOT / cfg).exists():
            print("Skip missing config:", cfg)
            continue
        benchmark(cfg, experiment_name, checkpoint=ckpt, debug=False, profile=True)

## 9. Read Metrics And Build Summary Tables

In [ ]:
CIFAR100_RUN_DIRS = [
    "runs_teacher_cifar100",
    "runs_baselines_cifar100",
    "runs_coc_baseline_cifar100",
    "runs_students_cifar100",
    "runs_accuracy_cifar100",
    "runs_proxy_cifar100",
    "runs_ablations_cifar100",
    "runs_kd_cifar100",
    "runs_pruning_accuracy_cifar100",
    "runs_pruned_accuracy_cifar100",
]


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def collect_train_metrics() -> pd.DataFrame:
    rows = []
    for run_root in CIFAR100_RUN_DIRS:
        for metrics_path in (ROOT / run_root).glob("*/metrics.jsonl"):
            records = read_jsonl(metrics_path)
            val_records = [r for r in records if "val_acc1" in r]
            if not val_records:
                continue
            best = max(val_records, key=lambda r: r["val_acc1"])
            test_records = [r for r in records if "test_acc1" in r]
            test_record = next((r for r in reversed(test_records) if r.get("epoch") == best.get("epoch")), test_records[-1] if test_records else {})
            train_records = [r for r in records if "train_acc1" in r]
            last_train = train_records[-1] if train_records else {}
            rows.append({
                "run_dir": str(metrics_path.parent.relative_to(ROOT)),
                "experiment": metrics_path.parent.name,
                "best_epoch": best.get("epoch"),
                "best_val_acc1": best.get("val_acc1"),
                "best_val_acc5": best.get("val_acc5"),
                "best_val_loss": best.get("val_loss"),
                "test_acc1": test_record.get("test_acc1"),
                "test_acc5": test_record.get("test_acc5"),
                "test_loss": test_record.get("test_loss"),
                "last_epoch": last_train.get("epoch"),
                "last_train_acc1": last_train.get("train_acc1"),
            })
    return pd.DataFrame(rows).sort_values("best_val_acc1", ascending=False) if rows else pd.DataFrame()


train_df = collect_train_metrics()
train_df

In [ ]:
def collect_benchmarks() -> pd.DataFrame:
    rows = []
    for path in (ROOT / BENCHMARK_OUTPUT).glob("*.json"):
        rec = json.loads(path.read_text(encoding="utf-8"))
        rec["benchmark_file"] = str(path.relative_to(ROOT))
        rows.append(rec)
    return pd.DataFrame(rows) if rows else pd.DataFrame()


bench_df = collect_benchmarks()
cols = [
    "config_id",
    "model_name",
    "params_total",
    "macs",
    "bops",
    "latency_ms_b1",
    "throughput_b16",
    "throughput_b64",
    "throughput_b128",
    "peak_memory_mb",
]
bench_df[[c for c in cols if c in bench_df.columns]] if not bench_df.empty else bench_df

## 10. Pareto Summary

In [ ]:
def attach_accuracy(bench: pd.DataFrame, train_metrics: pd.DataFrame) -> pd.DataFrame:
    if bench.empty:
        return bench
    out = bench.copy()
    out["acc1"] = None
    out["acc5"] = None
    if train_metrics.empty:
        return out
    acc1_map = dict(zip(train_metrics["experiment"], train_metrics["test_acc1"].fillna(train_metrics["best_val_acc1"])))
    acc5_source = train_metrics["test_acc5"].fillna(train_metrics["best_val_acc5"]) if "test_acc5" in train_metrics else None
    acc5_map = dict(zip(train_metrics["experiment"], acc5_source)) if acc5_source is not None else {}
    for idx, row in out.iterrows():
        name = row.get("config_id")
        out.at[idx, "acc1"] = acc1_map.get(name)
        out.at[idx, "acc5"] = acc5_map.get(name)
    return out


def pareto_frontier(df: pd.DataFrame) -> pd.DataFrame:
    required = ["acc1", "params_total", "latency_ms_b1", "peak_memory_mb"]
    if df.empty or any(c not in df.columns for c in required):
        return pd.DataFrame()
    valid = df.dropna(subset=required).copy()
    keep = []
    for i, row in valid.iterrows():
        dominated = False
        for j, other in valid.iterrows():
            if i == j:
                continue
            better_or_equal = (
                other["acc1"] >= row["acc1"]
                and other["params_total"] <= row["params_total"]
                and other["latency_ms_b1"] <= row["latency_ms_b1"]
                and other["peak_memory_mb"] <= row["peak_memory_mb"]
            )
            strictly_better = (
                other["acc1"] > row["acc1"]
                or other["params_total"] < row["params_total"]
                or other["latency_ms_b1"] < row["latency_ms_b1"]
                or other["peak_memory_mb"] < row["peak_memory_mb"]
            )
            if better_or_equal and strictly_better:
                dominated = True
                break
        if not dominated:
            keep.append(i)
    return valid.loc[keep].sort_values(["acc1", "latency_ms_b1"], ascending=[False, True])


combined_df = attach_accuracy(bench_df, train_df)
frontier_df = pareto_frontier(combined_df)

out_dir = ROOT / "results"
out_dir.mkdir(exist_ok=True)
if not combined_df.empty:
    combined_df.to_csv(out_dir / "cifar100_combined_training_benchmark.csv", index=False)
if not frontier_df.empty:
    frontier_df.to_csv(out_dir / "cifar100_pareto_frontier.csv", index=False)

frontier_df[[c for c in ["config_id", "acc1", "acc5", "params_total", "macs", "bops", "latency_ms_b1", "peak_memory_mb"] if c in frontier_df.columns]]

In [ ]:
if RUN_SUMMARY_REPORT:
    run_py([
        "tools/cifar100_report.py",
        "--runs",
        *CIFAR100_RUN_DIRS,
        "--benchmarks",
        BENCHMARK_OUTPUT,
        "--output",
        "results/cifar100_summary.csv",
    ])

if RUN_PARETO_REPORT:
    run_py(["tools/pareto_report.py", BENCHMARK_OUTPUT, "--output", "results/cifar100_pareto.md"])

## Recommended Execution Order

1. Run environment checks and smoke.
2. Enable `RUN_TEACHER` and train ResNet-18 on CIFAR-100.
3. Enable `RUN_COC_BASELINE` and train the CoC baseline with CIFAR-100 overrides.
4. Enable `RUN_STUDENT_CE` and train HBCC Tiny/Small plus HBCC-Accuracy Small/Medium.
5. Enable `RUN_PROXY_SEARCH`; use the table to choose top candidates.
6. Enable `RUN_ABLATIONS` for Hamming/LBPConv.
7. Enable `RUN_KD` after the CIFAR-100 teacher checkpoint exists.
8. Enable `RUN_PRUNING` after `runs_accuracy_cifar100/hbcc_accuracy_small_cifar100/best.pth` exists.
9. Enable `RUN_BENCHMARKS` and build final tables.

Do not treat proxy accuracy or untrained benchmark records as final scientific evidence.